In [ ]:
import collections
import importlib
import math
import os
import sys
from pathlib import Path
from typing import Callable, Dict, Optional, Sequence, Tuple, Union

import numpy as np
import torch
import torch.nn as nn
import torchvision
from diffusers.optimization import get_scheduler
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from rich import print
from tqdm.auto import tqdm

src_parent = os.path.abspath(os.path.join(os.path.dirname(os.getcwd())))
sys.path.append(src_parent)

import scripts
from scripts.image_dataset import PushTImageDataset, normalize_data, unnormalize_data
from scripts.model_config import TrainingConfig
from scripts.models import ConditionalUnet1D, VisionEncoder
from scripts.pusht_image_env import PushTImageEnv
from scripts.training_Inference import UnetTrainer


/home/thankgod/2025/msc_project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/home/thankgod/2025/msc_project/.venv/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package 

In [ ]:
dataset_path = Path.cwd().parent / "data/low_dim_pusht" / "replay_buffer.zarr"

# parameters
pred_horizon = 16
obs_horizon = 2
action_horizon = 8
#|o|o|                             observations: 2
#| |a|a|a|a|a|a|a|a|               actions executed: 8
#|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p| actions predicted: 16

# create dataset from file
dataset = PushTImageDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon
)
# save training data statistics (min, max) for each dim
stats = dataset.stats

# create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=64,
    num_workers=4,
    shuffle=True,
    # accelerate cpu-gpu transfer
    pin_memory=True,
    # don't kill worker process afte each epoch
    persistent_workers=True
)

# visualize data in batch
batch = next(iter(dataloader))
print("batch['image'].shape:", batch['image'].shape)
print("batch['agent_pos'].shape:", batch['agent_pos'].shape)
print("batch['action'].shape", batch['action'].shape)

NameError: name 'Path' is not defined

In [ ]:
# Encoder
# resnet18 efficientnet_b0 mobilenet_v2 mobilenet_v3_small mobilenet_v3_large
encoder = VisionEncoder()
vision_encoder, vision_feature_dim = encoder.get_model("resnet18")



# Example Input: Define the observation and agent position tensors
image = torch.zeros((1, obs_horizon, 3, 96, 96))
agent_pos = torch.zeros((1, obs_horizon, 2))
lowdim_obs_dim = agent_pos.shape[-1] # agent_pos is 2 dimensional
# observation feature depends on the selected model output
obs_dim = vision_feature_dim + lowdim_obs_dim
action_dim = 2


# Unet
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,  # input_dim (action_dim)
    global_cond_dim=obs_dim * obs_horizon,  # obs_dim=(512 + 2) * obs_horizon=2 -> 1028
)

nets = nn.ModuleDict({
    'vision_encoder': vision_encoder,
    'noise_pred_net': noise_pred_net
})

print(sum(p.numel() for p in nets.parameters()))